# Bronze route stream

Purpose: Read the raw Bronze stream, parse the event envelope, and route rows into:

- `aiops_bronze_vehicle_positions_records`
- `aiops_bronze_vehicle_positions_parse_quarantine`
- `aiops_bronze_vehicle_positions_quality_metrics`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, BooleanType
)

# Catalog / schema / paths
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

RAW_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.aiops_bronze_vehicle_positions_raw"
VALID_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.aiops_bronze_vehicle_positions_records"
QUARANTINE_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.aiops_bronze_vehicle_positions_parse_quarantine"
METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.aiops_bronze_vehicle_positions_quality_metrics"

VALID_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/bronze/records/hsl_vehicle_positions"
QUARANTINE_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/bronze/parse_quarantine/hsl_vehicle_positions"
METRICS_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/bronze/metrics/hsl_vehicle_positions_quality_metrics"

CHECKPOINT_VALID_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/bronze/checkpoints/records_hsl_vehicle_positions"
CHECKPOINT_QUARANTINE_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/bronze/checkpoints/parse_quarantine_hsl_vehicle_positions"
CHECKPOINT_METRICS_PATH = "abfss://lakehouse@streanmingdatasta.dfs.core.windows.net/external/hant-catalog/aiops/bronze/checkpoints/metrics_hsl_vehicle_positions"

TRIGGER_INTERVAL = "10 seconds"

RESET_VALID_TABLE = False
RESET_QUARANTINE_TABLE = False
RESET_METRICS_TABLE = False
RESET_VALID_CHECKPOINT = False
RESET_QUARANTINE_CHECKPOINT = False
RESET_METRICS_CHECKPOINT = False

## Object setup

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

# Stop routing queries before reset
for q in spark.streams.active:
    if q.name in {
        "aiops_bronze_vehicle_positions_records_route",
        "aiops_bronze_vehicle_positions_parse_quarantine_route",
        "aiops_bronze_vehicle_positions_metrics_route",
    }:
        q.stop()

if RESET_VALID_TABLE:
    spark.sql(f"DROP TABLE IF EXISTS {VALID_TABLE}")
    dbutils.fs.rm(VALID_PATH, True)

if RESET_QUARANTINE_TABLE:
    spark.sql(f"DROP TABLE IF EXISTS {QUARANTINE_TABLE}")
    dbutils.fs.rm(QUARANTINE_PATH, True)

if RESET_METRICS_TABLE:
    spark.sql(f"DROP TABLE IF EXISTS {METRICS_TABLE}")
    dbutils.fs.rm(METRICS_PATH, True)

if RESET_VALID_CHECKPOINT:
    dbutils.fs.rm(CHECKPOINT_VALID_PATH, True)

if RESET_QUARANTINE_CHECKPOINT:
    dbutils.fs.rm(CHECKPOINT_QUARANTINE_PATH, True)

if RESET_METRICS_CHECKPOINT:
    dbutils.fs.rm(CHECKPOINT_METRICS_PATH, True)

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {VALID_TABLE} (
    topic                  STRING,
    partition              INT,
    offset                 BIGINT,
    eventhub_enqueued_ts   TIMESTAMP,
    message_key            STRING,
    raw_json               STRING,
    bronze_ingest_ts       TIMESTAMP,
    ingest_date            DATE,
    parse_ok               BOOLEAN,
    parse_error            STRING,
    source                 STRING,
    producer_ingest_ts_utc STRING,
    mqtt_topic             STRING,
    mqtt_qos               INT,
    mqtt_retain            BOOLEAN,
    event_type             STRING,
    transport_mode         STRING
)
USING DELTA
PARTITIONED BY (ingest_date)
LOCATION "{VALID_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
    topic                  STRING,
    partition              INT,
    offset                 BIGINT,
    eventhub_enqueued_ts   TIMESTAMP,
    message_key            STRING,
    raw_json               STRING,
    bronze_ingest_ts       TIMESTAMP,
    ingest_date            DATE,
    parse_ok               BOOLEAN,
    parse_error            STRING,
    source                 STRING,
    producer_ingest_ts_utc STRING,
    mqtt_topic             STRING,
    mqtt_qos               INT,
    mqtt_retain            BOOLEAN,
    event_type             STRING,
    transport_mode         STRING
)
USING DELTA
PARTITIONED BY (ingest_date)
LOCATION "{QUARANTINE_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {METRICS_TABLE} (
    batch_id                  BIGINT,
    batch_ts                  TIMESTAMP,
    batch_date                DATE,
    total_rows                BIGINT,
    valid_rows                BIGINT,
    quarantined_rows          BIGINT,
    null_raw_json_rows        BIGINT,
    invalid_json_rows         BIGINT,
    missing_mqtt_topic_rows   BIGINT
)
USING DELTA
LOCATION "{METRICS_PATH}"
''')

DataFrame[]

## Minimal envelope schema for routing

In [0]:
envelope_schema = StructType([
    StructField("ingest_ts_utc", StringType(), True),
    StructField("source", StringType(), True),
    StructField("mqtt", StructType([
        StructField("topic", StringType(), True),
        StructField("qos", IntegerType(), True),
        StructField("retain", BooleanType(), True),
    ]), True),
    StructField("topic_parsed", StructType([
        StructField("event_type", StringType(), True),
        StructField("transport_mode", StringType(), True),
    ]), True),
])

## Read Bronze raw as a stream

In [0]:
bronze_raw_input = spark.readStream.table(RAW_TABLE)

## Parse and classify rows

In [0]:
bronze_enriched_stream = (
    bronze_raw_input
        .withColumn("parsed_envelope", F.from_json(F.col("raw_json"), envelope_schema))
        .withColumn(
            "parse_ok",
            F.col("raw_json").isNotNull()
            & F.col("parsed_envelope.source").isNotNull()
            & F.col("parsed_envelope.mqtt.topic").isNotNull()
        )
        .withColumn(
            "parse_error",
            F.when(F.col("raw_json").isNull(), F.lit("raw_json_is_null"))
             .when(F.col("parsed_envelope.source").isNull(), F.lit("invalid_envelope_json"))
             .when(F.col("parsed_envelope.mqtt.topic").isNull(), F.lit("missing_mqtt_topic"))
             .otherwise(F.lit(None).cast("string"))
        )
        .withColumn("source", F.col("parsed_envelope.source"))
        .withColumn("producer_ingest_ts_utc", F.col("parsed_envelope.ingest_ts_utc"))
        .withColumn("mqtt_topic", F.col("parsed_envelope.mqtt.topic"))
        .withColumn("mqtt_qos", F.col("parsed_envelope.mqtt.qos"))
        .withColumn("mqtt_retain", F.col("parsed_envelope.mqtt.retain"))
        .withColumn("event_type", F.col("parsed_envelope.topic_parsed.event_type"))
        .withColumn("transport_mode", F.col("parsed_envelope.topic_parsed.transport_mode"))
        .drop("parsed_envelope")
)

ROUTED_COLUMNS = [
    "topic",
    "partition",
    "offset",
    "eventhub_enqueued_ts",
    "message_key",
    "raw_json",
    "bronze_ingest_ts",
    "ingest_date",
    "parse_ok",
    "parse_error",
    "source",
    "producer_ingest_ts_utc",
    "mqtt_topic",
    "mqtt_qos",
    "mqtt_retain",
    "event_type",
    "transport_mode",
]

## Start valid-row routing query

In [0]:
valid_query = (
    bronze_enriched_stream
        .select(*ROUTED_COLUMNS)
        .writeStream
        .queryName("aiops_bronze_vehicle_positions_records_route")
        .format("delta")
        .outputMode("append")
        .trigger(processingTime=TRIGGER_INTERVAL)
        .option("checkpointLocation", CHECKPOINT_VALID_PATH)
        .toTable(VALID_TABLE)
)

print("AIOps Bronze records routing stream started.")
print("  Query name :", valid_query.name)
print("  Query ID   :", valid_query.id)
print("  Status     :", valid_query.status)

Valid routing stream started.
  Query name : bronze_vehicle_positions_valid_route
  Query ID   : 95386496-f56a-4d85-9554-b50646d29c5c
  Status     : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Start quarantine routing query

In [0]:
quarantine_query = (
    bronze_enriched_stream
        .filter(F.col("parse_ok") == False)
        .select(*ROUTED_COLUMNS)
        .writeStream
        .queryName("aiops_bronze_vehicle_positions_parse_quarantine_route")
        .format("delta")
        .outputMode("append")
        .trigger(processingTime=TRIGGER_INTERVAL)
        .option("checkpointLocation", CHECKPOINT_QUARANTINE_PATH)
        .toTable(QUARANTINE_TABLE)
)

print("AIOps Bronze parse quarantine stream started.")
print("  Query name :", quarantine_query.name)
print("  Query ID   :", quarantine_query.id)
print("  Status     :", quarantine_query.status)

Quarantine routing stream started.
  Query name : bronze_vehicle_positions_quarantine_route
  Query ID   : 4eb42a01-e5da-4240-9aac-578dd608b210
  Status     : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Start batch-level metrics query

In [0]:
def write_bronze_metrics(batch_df, batch_id: int) -> None:
    if batch_df.isEmpty():
        return

    metrics_df = (
        batch_df.agg(
            F.count("*").cast("bigint").alias("total_rows"),
            F.sum(F.when(F.col("parse_ok") == True, 1).otherwise(0)).cast("bigint").alias("valid_rows"),
            F.sum(F.when(F.col("parse_ok") == False, 1).otherwise(0)).cast("bigint").alias("quarantined_rows"),
            F.sum(F.when(F.col("parse_error") == "raw_json_is_null", 1).otherwise(0)).cast("bigint").alias("null_raw_json_rows"),
            F.sum(F.when(F.col("parse_error") == "invalid_envelope_json", 1).otherwise(0)).cast("bigint").alias("invalid_json_rows"),
            F.sum(F.when(F.col("parse_error") == "missing_mqtt_topic", 1).otherwise(0)).cast("bigint").alias("missing_mqtt_topic_rows"),
        )
        .withColumn("batch_id", F.lit(batch_id).cast("bigint"))
        .withColumn("batch_ts", F.current_timestamp())
        .withColumn("batch_date", F.to_date(F.current_timestamp()))
        .select(
            "batch_id",
            "batch_ts",
            "batch_date",
            "total_rows",
            "valid_rows",
            "quarantined_rows",
            "null_raw_json_rows",
            "invalid_json_rows",
            "missing_mqtt_topic_rows",
        )
    )

    (
        metrics_df.write
            .format("delta")
            .mode("append")
            .save(METRICS_PATH)
    )

In [0]:
metrics_query = (
    bronze_enriched_stream
        .writeStream
        .queryName("aiops_bronze_vehicle_positions_metrics_route")
        .foreachBatch(write_bronze_metrics)
        .trigger(processingTime=TRIGGER_INTERVAL)
        .option("checkpointLocation", CHECKPOINT_METRICS_PATH)
        .start()
)

print("AIOps Bronze metrics stream started.")
print("  Query name :", metrics_query.name)
print("  Query ID   :", metrics_query.id)
print("  Status     :", metrics_query.status)

Metrics routing stream started.
  Query name : bronze_vehicle_positions_metrics_route
  Query ID   : 2d3615a3-420d-4cf4-a08e-e8912c4a73fe
  Status     : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


## Monitor routing queries

In [0]:
for q in spark.streams.active:
    if q.name in {
        "aiops_bronze_vehicle_positions_records_route",
        "aiops_bronze_vehicle_positions_parse_quarantine_route",
        "aiops_bronze_vehicle_positions_metrics_route",
    }:
        print("NAME:", q.name)
        print("ID:", q.id)
        print("IS ACTIVE:", q.isActive)
        print("STATUS:", q.status)
        print("LAST PROGRESS:", q.lastProgress)
        print("EXCEPTION:", q.exception())
        print("-" * 80)

NAME: bronze_vehicle_positions_quarantine_route
ID: 4eb42a01-e5da-4240-9aac-578dd608b210
IS ACTIVE: True
STATUS: {'message': 'Writing offsets to log', 'isDataAvailable': False, 'isTriggerActive': True}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------
NAME: bronze_vehicle_positions_metrics_route
ID: 2d3615a3-420d-4cf4-a08e-e8912c4a73fe
IS ACTIVE: True
STATUS: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------
NAME: bronze_vehicle_positions_valid_route
ID: 95386496-f56a-4d85-9554-b50646d29c5c
IS ACTIVE: True
STATUS: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------
